# Queue: priority-1 seed runs (20k word-level DictaGen sets, seeds 43 and 44)

Runs the six training notebooks below one after another (each is identical to its seed-42 notebook except the training seed and the output repo).
Each run gets its own folder `~/work/queue_p1/<name>/` and pushes its model + results to its own public HF repo. A run whose HF repo already
holds `results.json` is skipped, so **re-running this notebook resumes** the queue. A failed run is reported and the queue goes on.

Start on the server: `./run.sh diffpen_words/queue_p1_seeds20k.ipynb` (inside tmux).

In [ ]:
import os, json, time, traceback
import papermill as pm
from huggingface_hub import HfApi
NBS = [
 "exp39_diffpenaug20k_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_uni10aug20k_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_join20k_diffpenaug_uni10aug_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_diffpenaug20k_hhdgeom_llmgen_seed44_colab.ipynb",
 "exp39_uni10aug20k_hhdgeom_llmgen_seed44_colab.ipynb",
 "exp39_join20k_diffpenaug_uni10aug_hhdgeom_llmgen_seed44_colab.ipynb"
]
HERE = os.getcwd(); BASE = os.path.expanduser('~/work/queue_p1'); os.makedirs(BASE, exist_ok=True)
api = HfApi()

def out_repo(nb):
    src = ''.join(''.join(c['source']) for c in json.load(open(os.path.join(HERE, nb)))['cells'])
    return src.split('OUT_REPO = "', 1)[1].split('"', 1)[0]

def done(repo):
    try: return 'results.json' in api.list_repo_files(repo)
    except Exception: return False

status = {}
for i, nb in enumerate(NBS, 1):
    repo = out_repo(nb); name = nb.replace('_colab.ipynb', '')
    if done(repo): print(f'[{i}/{len(NBS)}] {name}: already on HF, skipped', flush=True); status[name] = 'skipped (done)'; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f'[{i}/{len(NBS)}] {name} -> {repo}  started {time.strftime("%H:%M")}', flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, nb), os.path.join(wd, nb), cwd=wd, log_output=True, kernel_name='python3')
        status[name] = f'ok ({(time.time() - t0) / 3600:.2f} h)' if done(repo) else 'finished, but results.json not on HF'
    except Exception as e:
        status[name] = 'FAILED: ' + repr(e)[:300]; traceback.print_exc()
    print(f'[{i}/{len(NBS)}] {name}: {status[name]}', flush=True)
print('\n==== queue summary'); [print(f'{k:60s} {v}') for k, v in status.items()]

## Results on HF (greedy and beam 4 with the notebooks' own LM; the LM1 numbers for the paper are computed afterwards with eval_diffpen10k_vs_font10k.py)

In [ ]:
from huggingface_hub import hf_hub_download
for nb in NBS:
    repo = out_repo(nb)
    if not done(repo): print(f'{repo}: no results'); continue
    T = json.load(open(hf_hub_download(repo, 'results.json', force_download=True)))['final']['test_decoding']
    print(repo, {k.split(' (')[0]: round(100 * v['cer'], 2) for k, v in T.items()})